# GRU over consecutive typing attempts

This experiment reuses the audited Session 1 repetition-window protocol exactly. A row is one typing attempt; a length-10 window spans ten consecutive repetition IDs for one participant. These are not individual keystroke events. Session 2 is not used.

The split is chronological per participant: fit repetitions 1–70, embargo 71–79, and validation from 80 onward. The nine-repetition embargo is `window_length - 1`; windows are formed only after splitting and do not cross participant or repetition-gap boundaries. The scaler and label encoder are fit on fit-partition data only.

The GRU uses the same input windows, seed, hidden width, dropout, optimizer, learning rate, batch size, epoch limit, and early-stopping rule as the audited LSTM, changing the recurrent cell to GRU. This makes the GRU-vs-LSTM validation comparison more controlled than comparing either sequence model with single-attempt MLP results, though stochastic training and the single fixed split still limit conclusions.

In [ ]:
from pathlib import Path
import hashlib
import random

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder, StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA_PATH = PROJECT_DIR / 'data' / 'raw' / 'fixed-text.csv'
LSTM_MODEL_PATH = PROJECT_DIR / 'models' / 'deepkey_lstm_repetition_w10.keras'
MODEL_OUTPUT_PATH = PROJECT_DIR / 'models' / 'deepkey_gru_repetition_w10.keras'
METRICS_OUTPUT_PATH = PROJECT_DIR / 'results' / 'metrics' / 'gru_repetition_w10_validation.csv'
HISTORY_OUTPUT_PATH = PROJECT_DIR / 'results' / 'metrics' / 'gru_repetition_w10_training_history.csv'

WINDOW_LENGTH = 10
FIT_MAX_REPETITION = 70
VALIDATION_MIN_REPETITION = 80
GRU_UNITS = 64
DROPOUT_RATE = 0.20
LEARNING_RATE = 0.001
BATCH_SIZE = 256
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10

EXPECTED_FIT_WINDOW_IDS_SHA256 = 'fddb824bb58a983dfa3b4e6eef9905a215e53db82fce0dfa204530c82e599b85'
EXPECTED_VALIDATION_WINDOW_IDS_SHA256 = '2e4d0d0ddae0bd6fecb7f82e37385c75c73b2820f3f172e46489c9d552c5aeb7'

output_paths = [MODEL_OUTPUT_PATH, METRICS_OUTPUT_PATH, HISTORY_OUTPUT_PATH]
existing_outputs = [path for path in output_paths if path.exists()]
if existing_outputs:
    raise FileExistsError(f'Refusing to overwrite existing outputs: {existing_outputs}')

print('TensorFlow:', tf.__version__)
print('Project directory:', PROJECT_DIR)
print('Seed:', SEED, '| window length:', WINDOW_LENGTH)

In [ ]:
df_raw = pd.read_csv(RAW_DATA_PATH)
df_model = df_raw.copy()
exclusions = [('p020', 2, 4), ('p071', 1, 20), ('p087', 1, 8)]
for participant, session, repetition in exclusions:
    matching = (
        df_model['participant'].eq(participant)
        & df_model['session'].eq(session)
        & df_model['repetition'].eq(repetition)
    )
    if int(matching.sum()) != 1:
        raise ValueError(f'Expected exactly one documented exclusion: {(participant, session, repetition)}')
    df_model = df_model.loc[~matching].copy()

excluded_columns = {'participant', 'session', 'repetition', 'total time'}
feature_columns = [column for column in df_model.columns if column not in excluded_columns]
if len(df_model) != 19769 or len(feature_columns) != 46:
    raise ValueError(f'Unexpected corrected modeling dataset: {df_model.shape}; features={len(feature_columns)}')
if feature_columns != [column for column in df_raw.columns if column not in excluded_columns]:
    raise ValueError('Feature order does not match raw CSV order.')

df_session_1 = df_model.loc[df_model['session'].eq(1)].copy()
df_session_1['_raw_row_id'] = df_session_1.index
if len(df_session_1) != 9876 or df_session_1['participant'].nunique() != 99:
    raise ValueError('Unexpected Session 1 attempt or participant count.')
if df_session_1.duplicated(['participant', 'session', 'repetition']).any():
    raise ValueError('Duplicate participant/session/repetition keys found.')

fit_rows = df_session_1.loc[df_session_1['repetition'].le(FIT_MAX_REPETITION)].copy()
embargo_rows = df_session_1.loc[
    df_session_1['repetition'].between(FIT_MAX_REPETITION + 1, VALIDATION_MIN_REPETITION - 1)
].copy()
validation_rows = df_session_1.loc[df_session_1['repetition'].ge(VALIDATION_MIN_REPETITION)].copy()
if (len(fit_rows), len(embargo_rows), len(validation_rows)) != (6928, 891, 2057):
    raise ValueError('Chronological partition attempt counts differ from audited LSTM design.')
if VALIDATION_MIN_REPETITION - FIT_MAX_REPETITION - 1 < WINDOW_LENGTH - 1:
    raise ValueError('Boundary embargo is shorter than window_length - 1.')

print('Corrected Session 1 attempts:', len(df_session_1))
print('Fit / embargo / validation attempt rows:', len(fit_rows), len(embargo_rows), len(validation_rows))
print('Feature count:', len(feature_columns), '| participants:', df_session_1['participant'].nunique())
print('Session 2 is not included in model development.')

In [ ]:
fit_participants = set(fit_rows['participant'].unique())
validation_participants = set(validation_rows['participant'].unique())
if fit_participants != validation_participants or len(fit_participants) != 99:
    raise ValueError('Fit and validation partitions must both cover all 99 participants.')

X_fit_rows = fit_rows[feature_columns].apply(pd.to_numeric, errors='raise').to_numpy(dtype=np.float64)
X_validation_rows = validation_rows[feature_columns].apply(pd.to_numeric, errors='raise').to_numpy(dtype=np.float64)
if not np.isfinite(X_fit_rows).all() or not np.isfinite(X_validation_rows).all():
    raise ValueError('Non-finite values found in the feature matrix.')

scaler = StandardScaler()
X_fit_scaled = scaler.fit_transform(X_fit_rows).astype(np.float32)
X_validation_scaled = scaler.transform(X_validation_rows).astype(np.float32)
label_encoder = LabelEncoder().fit(fit_rows['participant'])
unseen_validation_labels = set(validation_rows['participant']) - set(label_encoder.classes_)
if unseen_validation_labels:
    raise ValueError(f'Validation contains unseen labels: {unseen_validation_labels}')

fit_scaled_by_row = dict(zip(fit_rows['_raw_row_id'].tolist(), X_fit_scaled))
validation_scaled_by_row = dict(zip(validation_rows['_raw_row_id'].tolist(), X_validation_scaled))

def build_windows(partition_rows, scaled_by_row):
    sequences, labels, window_row_ids = [], [], []
    ordered = partition_rows.sort_values(['participant', 'repetition'])
    for participant, group in ordered.groupby('participant', sort=True):
        row_ids = group['_raw_row_id'].tolist()
        repetitions = group['repetition'].astype(int).tolist()
        run_start = 0
        for position in range(1, len(group) + 1):
            at_run_end = position == len(group) or repetitions[position] != repetitions[position - 1] + 1
            if not at_run_end:
                continue
            run_ids = row_ids[run_start:position]
            for start in range(len(run_ids) - WINDOW_LENGTH + 1):
                window_ids = tuple(run_ids[start:start + WINDOW_LENGTH])
                sequences.append(np.stack([scaled_by_row[row_id] for row_id in window_ids]))
                labels.append(label_encoder.transform([participant])[0])
                window_row_ids.append(window_ids)
            run_start = position
    return np.asarray(sequences, dtype=np.float32), np.asarray(labels, dtype=np.int64), window_row_ids

def window_ids_sha256(window_row_ids):
    payload = '\n'.join(','.join(map(str, row_ids)) for row_ids in window_row_ids).encode('ascii')
    return hashlib.sha256(payload).hexdigest()

X_fit, y_fit, fit_window_row_ids = build_windows(fit_rows, fit_scaled_by_row)
X_validation, y_validation, validation_window_row_ids = build_windows(
    validation_rows, validation_scaled_by_row
)
fit_window_fingerprint = window_ids_sha256(fit_window_row_ids)
validation_window_fingerprint = window_ids_sha256(validation_window_row_ids)
if len(X_fit) != 6021 or len(X_validation) != 1166:
    raise ValueError('Window counts differ from audited LSTM design.')
if fit_window_fingerprint != EXPECTED_FIT_WINDOW_IDS_SHA256:
    raise ValueError('Fit window raw-row IDs differ from audited LSTM design.')
if validation_window_fingerprint != EXPECTED_VALIDATION_WINDOW_IDS_SHA256:
    raise ValueError('Validation window raw-row IDs differ from audited LSTM design.')

fit_raw_ids = {row_id for window in fit_window_row_ids for row_id in window}
validation_raw_ids = {row_id for window in validation_window_row_ids for row_id in window}
if fit_raw_ids.intersection(validation_raw_ids):
    raise AssertionError('A raw attempt is shared across fit and validation windows.')
for partition_rows, windows in ((fit_rows, fit_window_row_ids), (validation_rows, validation_window_row_ids)):
    participant_by_id = partition_rows.set_index('_raw_row_id')['participant'].to_dict()
    repetition_by_id = partition_rows.set_index('_raw_row_id')['repetition'].astype(int).to_dict()
    for row_ids in windows:
        if len({participant_by_id[row_id] for row_id in row_ids}) != 1:
            raise AssertionError('Window crosses participant boundaries.')
        repetitions = [repetition_by_id[row_id] for row_id in row_ids]
        if any(right != left + 1 for left, right in zip(repetitions, repetitions[1:])):
            raise AssertionError('Window crosses a missing repetition.')
if int(scaler.n_samples_seen_) != len(fit_rows):
    raise AssertionError('Scaler was not fit on exactly the fit-partition attempt rows.')
if len(label_encoder.classes_) != 99 or np.unique(y_fit).size != 99 or np.unique(y_validation).size != 99:
    raise AssertionError('Expected all 99 encoded participants in both window sets.')

print('Fit/validation window counts:', len(X_fit), len(X_validation))
print('Window shapes:', X_fit.shape, X_validation.shape)
print('Scaler fit rows:', int(scaler.n_samples_seen_), '| encoded labels: 0..', len(label_encoder.classes_) - 1)
print('Raw attempt overlap:', len(fit_raw_ids.intersection(validation_raw_ids)))
print('Audited ordered-window ID fingerprints matched.')

In [ ]:
gru_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(WINDOW_LENGTH, len(feature_columns))),
    tf.keras.layers.GRU(GRU_UNITS),
    tf.keras.layers.Dropout(DROPOUT_RATE),
    tf.keras.layers.Dense(len(label_encoder.classes_), activation='softmax'),
])
gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True,
)
history = gru_model.fit(
    X_fit, y_fit,
    validation_data=(X_validation, y_validation),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=2,
)

best_epoch = int(np.argmin(history.history['val_loss']) + 1)
gru_probabilities = gru_model.predict(X_validation, batch_size=BATCH_SIZE, verbose=0)
gru_predictions = gru_probabilities.argmax(axis=1)
gru_precision, gru_recall, gru_macro_f1, _ = precision_recall_fscore_support(
    y_validation, gru_predictions, average='macro', zero_division=0,
)
gru_metrics = {
    'accuracy': accuracy_score(y_validation, gru_predictions),
    'balanced_accuracy': balanced_accuracy_score(y_validation, gru_predictions),
    'macro_precision': gru_precision,
    'macro_recall': gru_recall,
    'macro_f1': gru_macro_f1,
}
history_frame = pd.DataFrame(history.history)
history_frame.insert(0, 'epoch', np.arange(1, len(history_frame) + 1))
history_frame['best_epoch_by_val_loss'] = best_epoch
print('GRU best epoch:', best_epoch)
print('GRU validation metrics:', {key: round(value, 6) for key, value in gru_metrics.items()})
print('Training history rows:', len(history_frame))

In [ ]:
lstm_model = tf.keras.models.load_model(LSTM_MODEL_PATH)
if lstm_model.input_shape != (None, WINDOW_LENGTH, len(feature_columns)):
    raise ValueError(f'Unexpected LSTM input shape: {lstm_model.input_shape}')
if lstm_model.output_shape != (None, len(label_encoder.classes_)):
    raise ValueError(f'Unexpected LSTM output shape: {lstm_model.output_shape}')
lstm_probabilities = lstm_model.predict(X_validation, batch_size=BATCH_SIZE, verbose=0)
lstm_predictions = lstm_probabilities.argmax(axis=1)
lstm_precision, lstm_recall, lstm_macro_f1, _ = precision_recall_fscore_support(
    y_validation, lstm_predictions, average='macro', zero_division=0,
)
lstm_metrics = {
    'accuracy': accuracy_score(y_validation, lstm_predictions),
    'balanced_accuracy': balanced_accuracy_score(y_validation, lstm_predictions),
    'macro_precision': lstm_precision,
    'macro_recall': lstm_recall,
    'macro_f1': lstm_macro_f1,
}

metric_names = ['accuracy', 'balanced_accuracy', 'macro_precision', 'macro_recall', 'macro_f1']
comparison = pd.DataFrame([
    {'model': 'lstm_repetition_window_10', **{key: lstm_metrics[key] for key in metric_names}},
    {'model': 'gru_repetition_window_10', **{key: gru_metrics[key] for key in metric_names}},
])
comparison['fit_windows'] = len(X_fit)
comparison['validation_windows'] = len(X_validation)
comparison['validation_protocol'] = 'Session 1 chronological per-participant; repetitions >=80; 9-repetition embargo'
print('Same-window GRU/LSTM validation comparison:')
print(comparison.to_string(index=False))

output_paths = [MODEL_OUTPUT_PATH, METRICS_OUTPUT_PATH, HISTORY_OUTPUT_PATH]
existing_outputs = [path for path in output_paths if path.exists()]
if existing_outputs:
    raise FileExistsError(f'Refusing to overwrite existing outputs: {existing_outputs}')

metrics_output = pd.DataFrame([{
    'model': 'gru_repetition_window_10',
    'window_length': WINDOW_LENGTH,
    'fit_max_repetition': FIT_MAX_REPETITION,
    'validation_min_repetition': VALIDATION_MIN_REPETITION,
    'boundary_embargo_repetitions': VALIDATION_MIN_REPETITION - FIT_MAX_REPETITION - 1,
    'seed': SEED,
    'fit_attempt_rows': len(fit_rows),
    'validation_attempt_rows': len(validation_rows),
    'fit_windows': len(X_fit),
    'validation_windows': len(X_validation),
    'scaler_fit_rows': int(scaler.n_samples_seen_),
    'fit_window_ids_sha256': fit_window_fingerprint,
    'validation_window_ids_sha256': validation_window_fingerprint,
    'gru_units': GRU_UNITS,
    'dropout_rate': DROPOUT_RATE,
    'learning_rate': LEARNING_RATE,
    'batch_size': BATCH_SIZE,
    'max_epochs': MAX_EPOCHS,
    'epochs_run': len(history_frame),
    'best_epoch': best_epoch,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    **{f'gru_validation_{key}': value for key, value in gru_metrics.items()},
    **{f'lstm_same_windows_validation_{key}': value for key, value in lstm_metrics.items()},
}])
MODEL_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
HISTORY_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
gru_model.save(MODEL_OUTPUT_PATH)
metrics_output.to_csv(METRICS_OUTPUT_PATH, index=False)
history_frame.to_csv(HISTORY_OUTPUT_PATH, index=False)
print('Saved GRU model:', MODEL_OUTPUT_PATH)
print('Saved validation metrics/comparison:', METRICS_OUTPUT_PATH)
print('Saved training history:', HISTORY_OUTPUT_PATH)
print('Session 2 was not accessed or evaluated.')